In [13]:
# 必要なモジュールをインポート
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from typing import Annotated
from typing_extensions import TypedDict
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

from langchain_core import messages

# ===== Stateクラスの定義 =====
class State(TypedDict):
    messages: Annotated[list, add_messages]

# ===== グラフの構築 =====
def build_graph(model_name):
    # ソースコードを記述
    # 検索ツールの定義
    tool = TavilySearchResults(max_results=2)
    tools = [tool]

    # グラフのインスタンスを作成
    graph_builder = StateGraph(State)
    # 言語モデルの定義
    llm = ChatOpenAI(model_name=model_name)
    # ツール定義の紐づけ
    llm_with_tools = llm.bind_tools(tools)

    # チャットボットノードの作成
    def chatbot(state: State):
        return {"messages": [llm_with_tools.invoke(state["messages"])]}
    # グラフにチャットボットノードを追加
    graph_builder.add_node("chatbot", chatbot)

    # ツールノードの作成
    tool_node = ToolNode(tools)
    # グラフにツールノードを追加
    graph_builder.add_node("tools", tool_node)

    # 条件付エッジの作成
    graph_builder.add_conditional_edges(
        "chatbot",
        tools_condition, # ツール呼出と判断したらツールノードを呼ぶ
    )
    # ツールが呼び出されるたびに、チャットボットに戻って次のステップを決定
    # ツールからチャットボットへの戻りエッジを作成
    graph_builder.add_edge("tools", "chatbot")
    # 開始ノードの指定
    graph_builder.set_entry_point("chatbot")

    return graph_builder

# ===== グラフ実行関数 =====
def stream_graph_updates(graph: StateGraph, user_input: str):
    # ソースコードを記述
    events = graph.stream(
        {"messages": [("user", user_input)]},
        {"configurable": {"thread_id": "1"}},
        stream_mode="values")
    # 結果をストリーミングで得る
    for event in events:
        # 末尾のメッセージを取得
        msg = event["messages"][-1]
        # ツールメッセージはスキップ
        if type(msg) is messages.tool.ToolMessage: continue
        # 空のメッセージはスキップ
        if msg.content == '': continue
        print(msg.content, flush=True)
        
# ===== メイン実行ロジック =====
# 環境変数の読み込み
load_dotenv("../.env")
os.environ['OPENAI_API_KEY'] = os.environ['API_KEY']

# モデル名
MODEL_NAME = "gpt-4o-mini" 

# グラフの作成
# ソースコードを記述
graph_builder = build_graph(MODEL_NAME)
# 記憶を持つ実行可能なステートグラフの作成
memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

# メインループ
# ソースコードを記述
# チャットボットのループ
while True:
    user_input = input("質問:")
    if user_input.strip()=="":
        print("ありがとうございました!")
        break
    stream_graph_updates(graph, user_input)
    print("")

こんにちは！
こんにちは！今日はどのようなことをお手伝いできますか？

1たす2は？
1たす2は3です。何か他に知りたいことがありますか？

台湾観光について検索結果を教えて
以下は台湾観光に関する情報です。

1. **交通部観光署 - 台湾観光情報ネット**
   - 台湾の観光情報を網羅した公式サイトです。観光スポット、食文化、イベントカレンダー、旅行サービスなど、台湾の観光に関する詳細な情報が掲載されています。
   - [公式サイトはこちら](https://jp.taiwan.net.tw/)

2. **台湾大学と観光**
   - 台湾には多くの大学があり、観光に関連する学部やプログラムもあります。特に、観光業に特化した教育機関では、実務に基づいたカリキュラムが提供されています。台湾を訪れる観光客に対するサービス向上のため、大学が果たす役割も注目されています。
   - 詳細は [こちらの記事](https://taiwan-talk.co.jp/taiwan-university/) で確認できます。

台湾には美しい風景や豊かな文化、美味しい食べ物がたくさんありますので、訪れる際はぜひ色々な場所を探検してみてください！他に知りたいことがあれば教えてください。

ありがとうございました!
